# ScriptShift pilot on bigger models (Kaggle)

Runs the same 150 pilot rows (30 items x 5 forms) on Qwen3-4B, Gemma-3-4B-it and Llama-3.1-8B-Instruct.
fp16, greedy decoding, one model at a time, memory freed between models. One jsonl per model, same format as `scripts/run_pilot.py`.

Before running: attach the private dataset with `pilot_items.csv`, add the `HF_TOKEN` secret, turn on GPU **T4 x2**,
and set `GIT_COMMIT` in the config cell. Score the downloaded jsonl files on your laptop with `scripts/score_pilot.py`.

In [ ]:
!pip install -q -U transformers accelerate huggingface_hub

In [ ]:
# Hugging Face login from the Kaggle secret named HF_TOKEN (Gemma and Llama are gated)
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret("HF_TOKEN"))

In [ ]:
import os, glob
# keep the (large) model cache off /kaggle/working, which is small and is what gets saved as output
os.environ.setdefault("HF_HOME", os.environ.get("PILOT_HF_CACHE", "/kaggle/tmp/hf"))

import csv, gc, hashlib, json, shutil, subprocess
from datetime import datetime
from pathlib import Path
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELS = json.loads(os.environ.get("PILOT_MODELS", json.dumps(
    ["Qwen/Qwen3-4B", "google/gemma-3-4b-it", "meta-llama/Llama-3.1-8B-Instruct"])))
ITEMS = os.environ.get("PILOT_ITEMS") or (glob.glob("/kaggle/input/**/pilot_items.csv", recursive=True) or [None])[0]
OUT_DIR = Path(os.environ.get("PILOT_OUT", "/kaggle/working/results/pilot"))
MAX_ROWS = int(os.environ.get("PILOT_MAX_ROWS", "0"))      # 0 = all rows
MAX_MEM = os.environ.get("PILOT_MAXMEM", "13GiB")          # per GPU; T4 has 15 GiB
DTYPE = {m: torch.float16 for m in MODELS}                 # fp16 for all. (If Gemma gives NaN / garbage, try bfloat16 for it.)
DTYPE["google/gemma-3-4b-it"] = torch.float32
SEED = 0
MAX_NEW_TOKENS = 256
GIT_COMMIT = "FILL_ME"   # run `git rev-parse HEAD` in the repo on your laptop and paste it here

assert ITEMS, "pilot_items.csv not found: attach your private dataset to this notebook"
GPUS = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print("GPUs:", GPUS, "| torch", torch.__version__, "| transformers", transformers.__version__)
print("items:", ITEMS)

In [ ]:
TOOLS_SYSTEM = (
    "You have access to the following functions:\n{tools}\n\n"
    "If a function matches the request, respond with only a JSON object of the form "
    "{{\"name\": <function-name>, \"arguments\": {{<argument-name>: <value>}}}} and nothing else. "
    "Otherwise answer normally."
)


def build_prompt(tok, query, tools):
    """Use the model's own chat template with tools. If that template does not render the tools
    (Gemma 3 has no tool support), fall back to putting the tool schema in a system message.
    Returns (prompt_text, tool_mode)."""
    msgs = [{"role": "user", "content": query}]
    prompt = tok.apply_chat_template(msgs, tools=tools, tokenize=False, add_generation_prompt=True,
                                     enable_thinking=False)
    if all(t["function"]["name"] in prompt for t in tools):
        return prompt, "template"
    system = TOOLS_SYSTEM.format(tools=json.dumps([t["function"] for t in tools], ensure_ascii=False, indent=1))
    prompt = tok.apply_chat_template([{"role": "system", "content": system}] + msgs, tokenize=False,
                                     add_generation_prompt=True)
    return prompt, "prompt_injected"


def revision_of(model_id):
    """commit hash of the downloaded snapshot (the folder name under snapshots/)"""
    from huggingface_hub import try_to_load_from_cache
    p = try_to_load_from_cache(model_id, "config.json")
    return Path(p).parent.name if p else None


def load_model(model_id):
    kw = dict(torch_dtype=DTYPE[model_id], device_map="auto",
              max_memory={i: MAX_MEM for i in range(torch.cuda.device_count())})
    try:
        return AutoModelForCausalLM.from_pretrained(model_id, **kw).eval()
    except (ValueError, KeyError):       # multimodal checkpoints (Gemma 3) on some transformers versions
        from transformers import Gemma3ForConditionalGeneration
        return Gemma3ForConditionalGeneration.from_pretrained(model_id, **kw).eval()


def free(model_id):
    gc.collect()
    torch.cuda.empty_cache()
    if not os.environ["HF_HOME"].startswith("/kaggle/"):      # only ever delete the Kaggle scratch cache
        return
    shutil.rmtree(Path(os.environ["HF_HOME"]) / "hub" / ("models--" + model_id.replace("/", "--")), ignore_errors=True)


def run_model(model_id, rows):
    torch.manual_seed(SEED)
    tok = AutoTokenizer.from_pretrained(model_id)
    model = load_model(model_id)
    first_prompt, tool_mode = build_prompt(tok, rows[0]["query"], json.loads(rows[0]["tools_json"]))
    meta = {
        "type": "meta", "model": model_id,
        "revision": revision_of(model_id),
        "dtype": str(DTYPE[model_id]).replace("torch.", ""),
        "decoding": {"do_sample": False, "max_new_tokens": MAX_NEW_TOKENS, "seed": SEED},
        "enable_thinking": False,
        "tool_mode": tool_mode,
        "prompt_template": ("model's own chat template with tools=<tools_json>, one user turn, add_generation_prompt=True"
                            if tool_mode == "template" else
                            "chat template has no tool support: tool schema in a system message (TOOLS_SYSTEM), one user turn"),
        "tools_system": TOOLS_SYSTEM if tool_mode == "prompt_injected" else None,
        "chat_template": tok.chat_template,
        "gpus": GPUS,
        "git": {"commit": GIT_COMMIT},
        "items_sha256": hashlib.sha256(Path(ITEMS).read_bytes()).hexdigest(),
        "torch": torch.__version__, "transformers": transformers.__version__,
        "time": datetime.now().isoformat(timespec="seconds"),
    }
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    out_path = OUT_DIR / f"{model_id.split('/')[-1].lower()}_{datetime.now():%Y%m%d_%H%M%S}.jsonl"
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(json.dumps(meta, ensure_ascii=False) + "\n")
        for i, r in enumerate(rows, 1):
            prompt, mode = build_prompt(tok, r["query"], json.loads(r["tools_json"]))
            assert mode == tool_mode
            inputs = tok(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
            with torch.no_grad():
                out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                     pad_token_id=tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id)
            raw = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)
            f.write(json.dumps({"type": "row", "item_id": r["item_id"], "form": r["form"],
                                "query": r["query"], "prompt": prompt, "raw_output": raw}, ensure_ascii=False) + "\n")
            f.flush()
            if i in (1, 2) or i % 25 == 0:
                print(f"  [{model_id}] {i}/{len(rows)}  {r['item_id']} {r['form']}: {raw[:140]!r}")
    del model
    return out_path

In [ ]:
rows = list(csv.DictReader(open(ITEMS, encoding="utf-8-sig", newline="")))
rows = [r for r in rows if r["query"].strip()]
if MAX_ROWS:
    rows = rows[:MAX_ROWS]
print(len(rows), "rows to run per model")

done = {}
for model_id in MODELS:
    print("=" * 20, model_id)
    try:
        done[model_id] = run_model(model_id, rows)
        print("saved", done[model_id])
    except Exception as e:                    # e.g. gated model whose license was not accepted: go on with the next one
        print("FAILED", model_id, "->", type(e).__name__, e)
    finally:
        free(model_id)
print(done)

In [ ]:
# one zip with all jsonl files: download it from the Output tab
shutil.make_archive("/kaggle/working/pilot_results", "zip", OUT_DIR)
print(os.listdir("/kaggle/working"))